### Display MTEB results
MTEB results are saved in .json files, one for each task. 
This notebook aggregates the results for multiple tasks and multiple models.

In [1]:
import os
os.getcwd()

'/Users/lena/Documents/GitHub/text_embedding/notebooks'

In [2]:
import mteb
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import regex as re


In [3]:
# get task type 
mteb.get_task("ArguAna").metadata.type

'Retrieval'

In [4]:
mteb.get_task("ArguAna").metadata.name

'ArguAna'

#### Analyze MTEB results of different TF-IDF configurations

In [5]:
data = {}
main_dir = "../MTEB/sparse_results/Tfidf"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
    # Identify model version from the folder structure
    model_version = os.path.basename(root)
    print(model_version)
        
    for file in files:
        # Skip unwanted files
        if file in {"model_meta.json"} or not file.endswith('.json'):
            continue
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                task_name = json_data.get("task_name")
                main_score = json_data.get("scores", {}).get("test", {})[0]["main_score"]
                main_score = round(main_score*100, 2)
                    
                if task_name and main_score is not None:
                    if task_name not in data:
                        data[task_name] = {}
                    data[task_name][model_version] = main_score
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


Tfidf
tfidf_log
svd300_log
svd50_log
svd_log_piecewise
svd_log_novocab
1.0
svd500_log
svd200_log
svd_log_old
rnd768_log
svd
svd_log
rnd100_log


In [6]:
df = pd.DataFrame.from_dict(data, orient='index')
df.index.name = "task_name"
df["task_types"] = [mteb.get_task(task).metadata.type for task in df.index]

In [7]:
task_selection = ["ArguAna", "ArxivClusteringP2P", "BiorxivClusteringP2P", "MedrxivClusteringP2P", "MindSmallReranking",
                 "RedditClusteringP2P", "SCIDOCS", "SciDocsRR", "StackExchangeClusteringP2P", "STS15", "STS16",
                 "STSBenchmark"]

df = df.loc[task_selection].sort_values("task_types")
df

,tfidf_log,svd300_log,svd50_log,svd_log_piecewise,svd_log_novocab,1.0,svd500_log,svd200_log,svd_log_old,rnd768_log,svd,svd_log,rnd100_log,task_types
task_name,,,,,,,,,,,,,,
ArxivClusteringP2P,35.28,40.71,39.77,39.97,41.73,26.72,41.15,40.59,41.73,28.91,36.95,40.34,9.02,Clustering
BiorxivClusteringP2P,26.44,33.87,33.53,34.02,33.80,20.56,33.40,33.63,33.80,18.80,28.47,33.93,4.01,Clustering
MedrxivClusteringP2P,20.82,29.53,29.64,29.87,30.00,19.30,29.12,29.68,30.00,18.66,28.62,29.81,10.96,Clustering
RedditClusteringP2P,40.32,38.37,34.26,33.99,45.96,31.55,40.64,36.67,45.96,32.07,39.83,34.53,11.42,Clustering
StackExchangeClusteringP2P,16.94,31.68,35.97,34.07,34.00,18.72,29.87,32.83,34.03,15.83,32.71,34.18,17.33,Clustering
MindSmallReranking,NaN,27.37,25.87,NaN,NaN,NaN,NaN,27.08,NaN,26.96,23.44,26.64,24.85,Reranking
SciDocsRR,62.34,58.63,48.30,NaN,NaN,62.30,60.47,56.47,NaN,61.64,28.35,52.69,52.38,Reranking
ArguAna,52.54,51.04,33.31,NaN,NaN,42.48,53.94,48.15,NaN,41.78,0.04,41.56,11.14,Retrieval
SCIDOCS,14.69,7.73,4.09,NaN,NaN,13.31,9.58,6.73,NaN,12.85,0.04,5.31,3.73,Retrieval


#### Figure 1 / Table 1: compare MTEB performance of different TF-IDF versions 
excluded models: 
- svd_log_piecewise
- svd_log_novocab
- svd_log_old
- svd

These are excluded bc either svd or svd and vocab are computed for the batch, not for the entire data in clustering tasks. This differentiation doesn't make sense for the other tasks, so it is disregarded

In [8]:
df_mteb = df.drop(["svd_log_piecewise", "svd_log_novocab", "svd_log_old", "svd"], axis=1)

model_order = ["1.0", "tfidf_log", "svd50_log", "svd_log", "svd200_log", "svd300_log", "svd500_log", "rnd100_log", "rnd768_log"]
df_mteb = df_mteb.reindex(columns=model_order)
df_mteb

,1.0,tfidf_log,svd50_log,svd_log,svd200_log,svd300_log,svd500_log,rnd100_log,rnd768_log
task_name,,,,,,,,,
ArxivClusteringP2P,26.72,35.28,39.77,40.34,40.59,40.71,41.15,9.02,28.91
BiorxivClusteringP2P,20.56,26.44,33.53,33.93,33.63,33.87,33.40,4.01,18.80
MedrxivClusteringP2P,19.30,20.82,29.64,29.81,29.68,29.53,29.12,10.96,18.66
RedditClusteringP2P,31.55,40.32,34.26,34.53,36.67,38.37,40.64,11.42,32.07
StackExchangeClusteringP2P,18.72,16.94,35.97,34.18,32.83,31.68,29.87,17.33,15.83
MindSmallReranking,NaN,NaN,25.87,26.64,27.08,27.37,NaN,24.85,26.96
SciDocsRR,62.30,62.34,48.30,52.69,56.47,58.63,60.47,52.38,61.64
ArguAna,42.48,52.54,33.31,41.56,48.15,51.04,53.94,11.14,41.78
SCIDOCS,13.31,14.69,4.09,5.31,6.73,7.73,9.58,3.73,12.85


In [10]:
df_mteb.to_latex(na_rep="-", float_format="%.2f")

'\\begin{tabular}{lrrrrrrrrr}\n\\toprule\n & 1.0 & tfidf_log & svd50_log & svd_log & svd200_log & svd300_log & svd500_log & rnd100_log & rnd768_log \\\\\ntask_name &  &  &  &  &  &  &  &  &  \\\\\n\\midrule\nArxivClusteringP2P & 26.72 & 35.28 & 39.77 & 40.34 & 40.59 & 40.71 & 41.15 & 9.02 & 28.91 \\\\\nBiorxivClusteringP2P & 20.56 & 26.44 & 33.53 & 33.93 & 33.63 & 33.87 & 33.40 & 4.01 & 18.80 \\\\\nMedrxivClusteringP2P & 19.30 & 20.82 & 29.64 & 29.81 & 29.68 & 29.53 & 29.12 & 10.96 & 18.66 \\\\\nRedditClusteringP2P & 31.55 & 40.32 & 34.26 & 34.53 & 36.67 & 38.37 & 40.64 & 11.42 & 32.07 \\\\\nStackExchangeClusteringP2P & 18.72 & 16.94 & 35.97 & 34.18 & 32.83 & 31.68 & 29.87 & 17.33 & 15.83 \\\\\nMindSmallReranking & - & - & 25.87 & 26.64 & 27.08 & 27.37 & - & 24.85 & 26.96 \\\\\nSciDocsRR & 62.30 & 62.34 & 48.30 & 52.69 & 56.47 & 58.63 & 60.47 & 52.38 & 61.64 \\\\\nArguAna & 42.48 & 52.54 & 33.31 & 41.56 & 48.15 & 51.04 & 53.94 & 11.14 & 41.78 \\\\\nSCIDOCS & 13.31 & 14.69 & 4.09 & 5.31

### Figure 2 / Table 2: compare TF-IDF with other models
other models considered:
- glove 
- sentenceBERT (mpnet)
- MPcrops
- simcse
- scincl

-> compare these to best performing TF-IDF model

